# 28. Self-RAG

Let the model reflect: is retrieval needed, is each passage relevant, is the answer supported, is it useful?

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Self-RAG teaches the model to reflect on its own process: *Do I even need to retrieve? Is this passage relevant? Is my answer supported by it? Is it useful?* The paper trains special reflection tokens; here we reproduce the same decisions with prompts so the idea is easy to see.

## How it works

1. **Retrieve?** decide if retrieval is needed
2. For each retrieved passage: **IsRel** - relevant?
3. Generate a candidate answer per relevant passage
4. **IsSup** - is the answer supported by the passage?
5. **IsUse** - usefulness 1-5
6. Pick the best-scoring candidate

```text
 question -> [Retrieve?] --no--> answer directly
                 | yes
              retrieve k -> [IsRel] filter -> generate per passage -> [IsSup][IsUse] -> best candidate
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 11:03:50.246000 8894 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


The original Self-RAG fine-tunes special reflection tokens. Here we reproduce the *behaviour* with prompts: `Retrieve?`, `IsRel`, `IsSup`, `IsUse`.

In [2]:
def need_retrieval(q):  return llm_yes(f"Would looking up documents help answer this question accurately?\n\nQuestion: {q}")
def is_rel(q, p):       return llm_yes(f"Question: {q}\n\nPassage: {p}\n\nIs the passage relevant to the question?")
def is_supported(a, p): return llm_yes(f"Passage: {p}\n\nAnswer: {a}\n\nIs the answer fully supported by the passage?")
def usefulness(q, a):
    out = llm(f"Rate how useful the answer is for the question from 1 to 5. Reply with the number only.\n\nQuestion: {q}\n\nAnswer: {a}")
    m = re.search(r"[1-5]", out)
    return int(m.group()) if m else 1

Generate one candidate answer per relevant passage and keep the best-scoring one.

In [3]:
def self_rag(query, k=4):
    if not need_retrieval(query):
        return "no retrieval", llm(query)
    candidates = []
    for h in store.search(query, k):
        if not is_rel(query, h["text"]):
            continue
        a = answer(query, [h["text"]])
        score = 2 * is_supported(a, h["text"]) + usefulness(query, a)   # supported answers win ties
        candidates.append((score, a, h["text"]))
    if not candidates:
        return "no relevant passage", llm(query)
    best = max(candidates, key=lambda c: c[0])
    return f"best of {len(candidates)} candidates (score {best[0]})", best[1]

for q in ["What does a reranker do?", "Write a haiku about autumn."]:
    how, response = self_rag(q)
    print(f"Q: {q}\n  [{how}]\n  {response[:300]}\n")

Q: What does a reranker do?
  [no retrieval]
  A reranker is a machine learning model that refines the ranking of a set of items, such as documents, images, or videos, based on their relevance to a specific query or task.

In general, a reranker takes the output from another model (such as a retriever or a generator) and adjusts its ranking to b



Q: Write a haiku about autumn.
  [no retrieval]
  Golden leaves descend
Crisp air whispers through the trees
Autumn's gentle hue



## Print the full candidate table
See every candidate and its reflection scores - this transparency is the point of Self-RAG.

In [4]:
import pandas as pd
def self_rag_table(query, k=4):
    rows = []
    for h in store.search(query, k):
        rel = is_rel(query, h["text"])
        if rel:
            a = answer(query, [h["text"]])
            rows.append({"chunk": h["idx"], "relevant": True, "supported": is_supported(a, h["text"]), "useful(1-5)": usefulness(query, a), "answer": a[:70]})
        else:
            rows.append({"chunk": h["idx"], "relevant": False, "supported": None, "useful(1-5)": None, "answer": "-"})
    return pd.DataFrame(rows)

self_rag_table("What does a reranker do?")

,chunk,relevant,supported,useful(1-5),answer
0,6,True,True,4.0,"According to the context, a reranker is a cros..."
1,7,False,None,NaN,-
2,9,False,None,NaN,-
3,16,False,None,NaN,-


## Does it route correctly?
The first reflection (`Retrieve?`) should skip retrieval for questions that don't need documents.

In [5]:
for q in ["What does HyDE stand for?", "Write a haiku about autumn.", "What is 7 times 8?", "What is semantic caching?"]:
    print(f"retrieve={need_retrieval(q)!s:5}  {q}")

retrieve=False  What does HyDE stand for?
retrieve=False  Write a haiku about autumn.


retrieve=False  What is 7 times 8?


retrieve=False  What is semantic caching?


## Compare against plain RAG

In [6]:
compare_pipelines({"plain RAG": lambda q: rag(q, store, 3)[0], "Self-RAG (prompted)": lambda q: self_rag(q)[1]}, n=4)

,keyword_recall,judge_correct,avg_seconds
pipeline,,,
plain RAG,0.79,1.0,0.1
Self-RAG (prompted),0.03,0.0,10.0


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Skips retrieval when unnecessary | Many LLM calls (k+ per query) |
| Filters irrelevant passages | Prompted reflection is weaker than trained reflection tokens |
| Picks the best-supported candidate | Slow without parallelism |

## When to use it

Quality-critical applications where latency and cost are secondary. For production-grade Self-RAG, use the fine-tuned models from the paper.

## Experiments to try

- Run the per-passage branches in parallel.
- Add a fallback when no candidate is supported.
- Weight the scores differently (support vs usefulness).

## Key takeaways

- Reflection = ask yourself questions at each step.
- Generate multiple candidates, choose by evidence.
- Cost scales with k.

---
**Next:** [29. GraphRAG](../29_graph_rag/graph_rag.ipynb)